# Data assimilation in the latent space: Particle filter

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Open prior and observation datasets

In [ ]:
import xarray as xr
import numpy as np

fname_obs = "data/20181224_1800_Meteosat-11_Etna_VPRoutput.nc"

prior = xr.open_dataset("output/ETNA2018/prior.nc")
obs = xr.open_dataset(fname_obs)

In [ ]:
nens = prior.sizes['ens']
latent_dim = prior.sizes['latent_dim']
print(f"Ensemble size: {nens} --- Latent space size: {latent_dim}")

## Get observations

In [ ]:
valid = (obs["plume_mask"].values > 0) & (obs["mass"].values > 0)
y_obs = obs["mass"].values[valid].astype(np.float64)
lat_obs = obs["latitude"].values[valid].astype(np.float64)
lon_obs = obs["longitude"].values[valid].astype(np.float64)

In [ ]:
nobs = len(y_obs)
print(f"Number of observations: {nobs}")

In [ ]:
# Required for interpolations
lat_points = xr.DataArray(lat_obs, dims="obs")
lon_points = xr.DataArray(lon_obs, dims="obs")

## Get the prior ensemble

In [ ]:
Z = prior["z"].values.astype(np.float64)
X = prior["samples"]

Y_xr = X.interp(
    lat=lat_points,
    lon=lon_points,
    method="linear"
)
Y = Y_xr.values.astype(np.float64)

print("Y shape:", Y.shape)

In [ ]:
print("NaNs in Y:", np.isnan(Y).sum())

## Observation-error covariance

In [ ]:
# 50% error observation is assumed with a minimim of
# 0.1 g/m2 (typical satellite detection limit)
obs_error_fraction = 0.50
obs_error_std = obs_error_fraction * y_obs
obs_error_std[obs_error_std<0.1] = 0.1 # Apply a minimum error (satellite detection limit)

print("error std range:", obs_error_std.min(), obs_error_std.max())

## Check dimensions

In [ ]:
assert Z.shape == (nens, latent_dim), "Wrong dimensions for Z"
assert Y.shape == (nens, nobs), "Wrong dimensions for Y"
assert y_obs.shape == (nobs,), "Wrong dimensions for y_obs"

## Algorithm: Regularized SIR Particle Filter

#### Normalized particle-filter weights

For each prior particle, the weight is proportional to the likelihood of the observations given the particle's predicted observations. The weights are normalized to sum to one.

In [ ]:
def particle_weights(Y, y_obs, sigma_obs):
    """
    Compute normalized particle-filter weights.

    Parameters
    ----------
    Y : ndarray, shape (nens, nobs)
        Observation-space ensemble.
    y_obs : ndarray, shape (nobs,)
        Observations.
    sigma_obs : ndarray or float
        Observation standard deviation(s).

    Returns
    -------
    weights : ndarray, shape (nens,)
        Normalized particle weights.
    log_weights : ndarray, shape (nens,)
        Unnormalized log-likelihoods.
    """

    # Innovation for every particle
    innovation = Y - y_obs[None, :]

    # Mahalanobis distance for diagonal R
    log_weights = -0.5 * np.sum(
        (innovation / sigma_obs) ** 2,
        axis=1
    )

    # Numerically stable normalization
    log_weights -= np.max(log_weights)
    weights = np.exp(log_weights)
    weights /= np.sum(weights)

    return weights, log_weights

In [ ]:
weights, log_weights = particle_weights(Y,y_obs,obs_error_std)

In [ ]:
print("min weight:", weights.min())
print("max weight:", weights.max())
print("sum weights:", weights.sum())

In [ ]:
N_eff = 1.0 / np.sum(weights**2)

print("Effective sample size:", N_eff)
print("N_eff / nens:", N_eff / len(weights))

#### Resampling
The resampling step takes the weighted particles and generates a new ensemble of the same size by duplicating particles with high weights and discarding particles with low weights

In [ ]:
def systematic_resample(weights, rng=None):
    """
    Systematic resampling.

    Parameters
    ----------
    weights : ndarray, shape (nens,)
        Normalized particle weights. Must sum to 1.
    rng : np.random.Generator, optional
        Random number generator.

    Returns
    -------
    indices : ndarray, shape (nens,)
        Indices of resampled particles.
    """
    if rng is None:
        rng = np.random.default_rng()

    weights = np.asarray(weights)
    n = len(weights)

    # Cumulative distribution
    cdf = np.cumsum(weights)
    cdf[-1] = 1.0  # avoid numerical issues

    # One random offset, then equally spaced points
    u0 = rng.uniform(0.0, 1.0 / n)
    u = u0 + np.arange(n) / n

    # Find corresponding particles
    indices = np.searchsorted(cdf, u)

    return indices

In [ ]:
rng = np.random.default_rng(42)

indices = systematic_resample(weights, rng)

Z_resampled = Z[indices]

In [ ]:
unique, counts = np.unique(indices, return_counts=True)

print("Number of unique particles:", len(unique))
print("Fraction unique:", len(unique) / len(Z))

order = np.argsort(counts)[::-1]

for i in order[:10]:
    print(unique[i], counts[i], counts[i] / len(Z))

#### Covariance

In [ ]:
prior_mean = np.mean(Z, axis=0)
prior_cov = np.cov(Z, rowvar=False)

print("Prior std:")
print(np.sqrt(np.diag(prior_cov)))

print("Weighted std:")
print(np.sqrt(np.diag(z_cov)))

#### Kernel perturbation
Add a small random perturbation to the resampled particles in order to reduce particle degeneracy: without it, resampling can produce many identical particles, reducing ensemble diversity.

In [ ]:
regularization_factor = 0.2

Sigma_reg = prior_cov * regularization_factor**2

# Ensure exact symmetry
Sigma_reg = 0.5 * (Sigma_reg + Sigma_reg.T)

# Cholesky factor
jitter = 1e-10 * np.trace(Sigma_reg) / latent_dim

L = np.linalg.cholesky(Sigma_reg + jitter * np.eye(latent_dim))

# Generate correlated perturbations
noise = np.random.randn(nens, latent_dim)
perturbation = noise @ L.T

# Regularized posterior
Z_analysis = Z_resampled + perturbation

In [ ]:
print("Z_resampled std:", Z_resampled.std(axis=0))
print("Z_posterior std:", Z_analysis.std(axis=0))

## Summary

In [ ]:
Pz_prior = np.cov(Z, rowvar=False)
Pz_analysis = np.cov(Z_analysis, rowvar=False)

In [ ]:
variance_prior = np.trace(Pz_prior)
variance_analysis = np.trace(Pz_analysis)

print("Total prior variance:   ", variance_prior)
print("Total analysis variance:", variance_analysis)
print("Variance ratio:",variance_analysis / variance_prior)

## Save posterior

In [ ]:
posterior_latent = xr.Dataset(
    data_vars={
        "z": (
            ("ens", "latent_dim"),
            Z_analysis.astype(np.float32)
        ),
        "z_mean": (
            ("latent_dim",),
            Z_analysis.mean(axis=0).astype(np.float32)
        ),
    },
    coords={
        "ens": np.arange(nens),
        "latent_dim": np.arange(latent_dim),
    },
)

posterior_latent.to_netcdf(output_dir / "posterior-latent-pf.nc")